# Transform & Validate

Notebook ini membersihkan data dimensi dan fakta, menstandarkan tipe data, memvalidasi primary key dan foreign key, lalu menyimpan hasil ETL ke `data/processed`.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 120)

## Konfigurasi Path dan File

In [2]:
PROJECT_ROOT = Path.cwd() if (Path.cwd() / 'data' / 'raw').exists() else Path.cwd().parent
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
VALIDATION_DIR = PROJECT_ROOT / 'data' / 'validation'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)

FACT_SOURCE_FILE = 'fact_transaksi_layanan_fix.csv'

print('Raw data folder:', RAW_DIR.resolve())
print('Processed folder:', PROCESSED_DIR.resolve())
print('Validation folder:', VALIDATION_DIR.resolve())
print('Fact source:', FACT_SOURCE_FILE)

Raw data folder: C:\Codingan Kuliah\SEMESTER 6\Data Warehouse\atc-data-warehouse\data\raw
Processed folder: C:\Codingan Kuliah\SEMESTER 6\Data Warehouse\atc-data-warehouse\data\processed
Validation folder: C:\Codingan Kuliah\SEMESTER 6\Data Warehouse\atc-data-warehouse\data\validation
Fact source: fact_transaksi_layanan_fix.csv


## Extract Ulang Data Mentah

Data dibaca dengan `dtype=str` supaya format teks seperti nomor HP tidak berubah dan angka 0 di depan tetap aman.

In [3]:
dim_channel = pd.read_csv(RAW_DIR / 'dim_channel_pembelian.csv', dtype=str)
dim_lokasi = pd.read_csv(RAW_DIR / 'dim_lokasi.csv', dtype=str)
dim_pelanggan = pd.read_csv(RAW_DIR / 'dim_pelanggan.csv', dtype=str)
dim_produk = pd.read_csv(RAW_DIR / 'dim_produk.csv', dtype=str)
dim_status = pd.read_csv(RAW_DIR / 'dim_status_transaksi.csv', dtype=str)
dim_waktu = pd.read_csv(RAW_DIR / 'dim_waktu.csv', dtype=str)
fact_transaksi = pd.read_csv(RAW_DIR / FACT_SOURCE_FILE, dtype=str)

raw_tables = {
    'dim_channel_pembelian': dim_channel,
    'dim_lokasi': dim_lokasi,
    'dim_pelanggan': dim_pelanggan,
    'dim_produk': dim_produk,
    'dim_status_transaksi': dim_status,
    'dim_waktu': dim_waktu,
    'fact_transaksi_layanan': fact_transaksi,
}

for table_name, df in raw_tables.items():
    print(f'{table_name}: {df.shape[0]:,} baris, {df.shape[1]:,} kolom')

dim_channel_pembelian: 6 baris, 3 kolom
dim_lokasi: 98 baris, 7 kolom
dim_pelanggan: 1,000 baris, 7 kolom
dim_produk: 50 baris, 6 kolom
dim_status_transaksi: 5 baris, 3 kolom
dim_waktu: 1,096 baris, 6 kolom
fact_transaksi_layanan: 25,926 baris, 14 kolom


## Helper Cleaning

In [4]:
def clean_text(series):
    return (
        series.astype('string')
        .str.strip()
        .replace({'': pd.NA, 'nan': pd.NA, 'None': pd.NA, 'NULL': pd.NA, 'NaN': pd.NA})
    )


def to_int(series):
    cleaned = clean_text(series).str.replace(r'[^0-9\-]', '', regex=True)
    return pd.to_numeric(cleaned, errors='coerce').astype('Int64')


def to_float(series):
    cleaned = clean_text(series).str.replace(r'[^0-9\.\-]', '', regex=True)
    return pd.to_numeric(cleaned, errors='coerce')


def normalize_status_aktif(series):
    normalized = clean_text(series).str.lower()
    true_values = {'1', 'true', 't', 'yes', 'y', 'aktif'}
    false_values = {'0', 'false', 'f', 'no', 'n', 'tidak aktif', 'nonaktif'}
    return normalized.map(lambda value: True if value in true_values else False if value in false_values else pd.NA).astype('boolean')


def fill_unknown_text(df, columns):
    for column in columns:
        df[column] = clean_text(df[column]).fillna('Tidak Diketahui')
    return df


def validate_pk(df, table_name, pk_column):
    return {
        'table_name': table_name,
        'pk_column': pk_column,
        'row_count': len(df),
        'null_pk_count': int(df[pk_column].isna().sum()),
        'duplicate_pk_count': int(df[pk_column].duplicated().sum()),
        'duplicate_row_count': int(df.duplicated().sum()),
        'total_missing_values': int(df.isna().sum().sum()),
    }

## Transform Tabel Dimensi

In [5]:
# Dim channel pembelian
dim_channel_clean = dim_channel.copy()
dim_channel_clean['id_channel'] = to_int(dim_channel_clean['id_channel'])
dim_channel_clean = fill_unknown_text(dim_channel_clean, ['nama_channel', 'jenis_channel'])
dim_channel_clean = dim_channel_clean.drop_duplicates(subset=['id_channel']).sort_values('id_channel')

# Dim lokasi
dim_lokasi_clean = dim_lokasi.copy()
dim_lokasi_clean['id_lokasi'] = to_int(dim_lokasi_clean['id_lokasi'])
dim_lokasi_clean = fill_unknown_text(
    dim_lokasi_clean,
    ['site_id', 'kelurahan', 'kecamatan', 'kabupaten_kota', 'provinsi', 'regional'],
)
dim_lokasi_clean = dim_lokasi_clean.drop_duplicates(subset=['id_lokasi']).sort_values('id_lokasi')

# Dim pelanggan
dim_pelanggan_clean = dim_pelanggan.copy()
dim_pelanggan_clean['id_pelanggan'] = to_int(dim_pelanggan_clean['id_pelanggan'])
dim_pelanggan_clean['nomor_hp'] = clean_text(dim_pelanggan_clean['nomor_hp']).str.replace(r'\D', '', regex=True)
dim_pelanggan_clean['nomor_hp'] = dim_pelanggan_clean['nomor_hp'].apply(lambda value: f'0{value}' if pd.notna(value) and not str(value).startswith('0') else value)
dim_pelanggan_clean['nama_pelanggan'] = clean_text(dim_pelanggan_clean['nama_pelanggan']).fillna('Tidak Diketahui')
dim_pelanggan_clean = fill_unknown_text(dim_pelanggan_clean, ['jenis_layanan', 'jenis_kartu'])
dim_pelanggan_clean['tanggal_registrasi'] = pd.to_datetime(dim_pelanggan_clean['tanggal_registrasi'], errors='coerce').dt.date
dim_pelanggan_clean['status_aktif'] = normalize_status_aktif(dim_pelanggan_clean['status_aktif']).fillna(False)
dim_pelanggan_clean = dim_pelanggan_clean.drop_duplicates(subset=['id_pelanggan']).sort_values('id_pelanggan')

# Dim produk
dim_produk_clean = dim_produk.copy()
dim_produk_clean['id_produk'] = to_int(dim_produk_clean['id_produk'])
dim_produk_clean['harga'] = to_float(dim_produk_clean['harga']).fillna(0)
dim_produk_clean = fill_unknown_text(dim_produk_clean, ['kode_produk', 'nama_produk', 'jenis_produk', 'kategori_produk'])
dim_produk_clean = dim_produk_clean.drop_duplicates(subset=['id_produk']).sort_values('id_produk')

# Dim status transaksi
dim_status_clean = dim_status.copy()
dim_status_clean['id_status'] = to_int(dim_status_clean['id_status'])
dim_status_clean = fill_unknown_text(dim_status_clean, ['kode_status', 'deskripsi_status'])
dim_status_clean = dim_status_clean.drop_duplicates(subset=['id_status']).sort_values('id_status')

# Dim waktu
dim_waktu_clean = dim_waktu.copy()
dim_waktu_clean['id_waktu'] = to_int(dim_waktu_clean['id_waktu'])
dim_waktu_clean['tanggal_lengkap'] = pd.to_datetime(dim_waktu_clean['tanggal_lengkap'], errors='coerce')
dim_waktu_clean['tahun'] = to_int(dim_waktu_clean['tahun']).fillna(dim_waktu_clean['tanggal_lengkap'].dt.year.astype('Int64'))
dim_waktu_clean = fill_unknown_text(dim_waktu_clean, ['hari', 'bulan', 'kuartal'])
dim_waktu_clean['tanggal_lengkap'] = dim_waktu_clean['tanggal_lengkap'].dt.date
dim_waktu_clean = dim_waktu_clean.drop_duplicates(subset=['id_waktu']).sort_values('id_waktu')

dimension_tables = {
    'dim_channel_pembelian': dim_channel_clean,
    'dim_lokasi': dim_lokasi_clean,
    'dim_pelanggan': dim_pelanggan_clean,
    'dim_produk': dim_produk_clean,
    'dim_status_transaksi': dim_status_clean,
    'dim_waktu': dim_waktu_clean,
}

for table_name, df in dimension_tables.items():
    print(f'{table_name}: {df.shape[0]:,} baris, {df.shape[1]:,} kolom')
    display(df.head())

dim_channel_pembelian: 6 baris, 3 kolom


,id_channel,nama_channel,jenis_channel
0,1,MyTelkomsel,Digital
1,2,GraPARI,Fisik
2,3,Minimarket,Fisik
3,4,Website_Telkomsel,Digital
4,5,Mitra Dealer,Fisik


dim_lokasi: 98 baris, 7 kolom


,id_lokasi,site_id,kelurahan,kecamatan,kabupaten_kota,provinsi,regional
0,1,SITE-0001,Menteng,Menteng,Jakarta Pusat,DKI Jakarta,Regional Jawa
1,2,SITE-0002,Gambir,Gambir,Jakarta Pusat,DKI Jakarta,Regional Jawa
2,3,SITE-0003,Tanah Abang,Tanah Abang,Jakarta Pusat,DKI Jakarta,Regional Jawa
3,4,SITE-0004,Kemayoran,Kemayoran,Jakarta Pusat,DKI Jakarta,Regional Jawa
4,5,SITE-0005,Senen,Senen,Jakarta Pusat,DKI Jakarta,Regional Jawa


dim_pelanggan: 1,000 baris, 7 kolom


...

,id_produk,kode_produk,nama_produk,jenis_produk,kategori_produk,harga
0,1,PRD-0001,1 GB Harian,Paket Data,Harian,2000
1,2,PRD-0002,2 GB HARIAN,Paket Data,Harian,3500
2,3,PRD-0003,5 GB Harian,Paket Data,Harian,7000
3,4,PRD-0004,1 GB Mingguan,Paket Data,Mingguan,9000
4,5,PRD-0005,3 GB Mingguan,Paket Data,Mingguan,20000


dim_status_transaksi: 5 baris, 3 kolom


,id_status,kode_status,deskripsi_status
0,1,ST-001,Sukses
1,2,ST-002,Gagal
2,3,ST-003,Pending
3,4,ST-004,REFUND
4,5,ST-005,Dibatalkan


dim_waktu: 1,096 baris, 6 kolom


,id_waktu,tanggal_lengkap,hari,bulan,kuartal,tahun
0,1,2022-01-01,Sabtu,Januari,Q1,2022
1,2,2022-01-02,Minggu,Januari,Q1,2022
2,3,2022-01-03,Senin,Januari,Q1,2022
3,4,2022-01-04,Selasa,Januari,Q1,2022
4,5,2022-01-05,Rabu,Januari,Q1,2022


## Transform Tabel Fakta

In [6]:
fact_clean = fact_transaksi.copy()

if 'Unnamed: 0' in fact_clean.columns:
    fact_clean = fact_clean.drop(columns=['Unnamed: 0'])

id_columns = ['id_fakta', 'id_waktu', 'id_pelanggan', 'id_produk', 'id_lokasi', 'id_channel', 'id_status']
for column in id_columns:
    fact_clean[column] = to_int(fact_clean[column])

fact_clean['nomor_transaksi'] = clean_text(fact_clean['nomor_transaksi']).fillna('Tidak Diketahui')
fact_clean['jumlah_pembelian'] = to_int(fact_clean['jumlah_pembelian']).fillna(0)
fact_clean['total_harga'] = to_float(fact_clean['total_harga']).fillna(0)
fact_clean['jumlah_data_mb'] = to_float(fact_clean['jumlah_data_mb']).fillna(0)
fact_clean['durasi_telpon_menit'] = to_int(fact_clean['durasi_telpon_menit']).fillna(0)
fact_clean['jumlah_sms'] = to_int(fact_clean['jumlah_sms']).fillna(0)

fact_clean = fact_clean.drop_duplicates(subset=['id_fakta']).sort_values('id_fakta')

print(f'fact_transaksi_layanan setelah cleaning awal: {fact_clean.shape[0]:,} baris, {fact_clean.shape[1]:,} kolom')
display(fact_clean.head())

fact_transaksi_layanan setelah cleaning awal: 25,926 baris, 13 kolom


,id_fakta,id_waktu,id_pelanggan,id_produk,id_lokasi,id_channel,id_status,nomor_transaksi,jumlah_pembelian,total_harga,jumlah_data_mb,durasi_telpon_menit,jumlah_sms
0,4,671,243,34,14,3,1,TRX006710000004,1,200000.0,44081.52,527,223
1,5,858,793,34,68,1,1,TRX008580000005,3,600000.0,12863.57,126,372
2,6,603,525,36,37,6,1,TRX006030000006,3,450000.0,37873.83,139,147
3,9,79,18,32,8,2,1,TRX000790000009,3,285000.0,14917.96,582,213
4,15,95,798,40,88,2,1,TRX000950000015,1,10000.0,27047.78,904,259


## Validasi Primary Key

In [7]:
pk_config = {
    'dim_channel_pembelian': ('id_channel', dim_channel_clean),
    'dim_lokasi': ('id_lokasi', dim_lokasi_clean),
    'dim_pelanggan': ('id_pelanggan', dim_pelanggan_clean),
    'dim_produk': ('id_produk', dim_produk_clean),
    'dim_status_transaksi': ('id_status', dim_status_clean),
    'dim_waktu': ('id_waktu', dim_waktu_clean),
    'fact_transaksi_layanan': ('id_fakta', fact_clean),
}

table_validation = [validate_pk(df, table_name, pk_column) for table_name, (pk_column, df) in pk_config.items()]
table_validation_df = pd.DataFrame(table_validation)
table_validation_df['is_pk_valid'] = (table_validation_df['null_pk_count'] == 0) & (table_validation_df['duplicate_pk_count'] == 0)

display(table_validation_df)

,table_name,pk_column,row_count,null_pk_count,duplicate_pk_count,duplicate_row_count,total_missing_values,is_pk_valid
0,dim_channel_pembelian,id_channel,6,0,0,0,0,True
1,dim_lokasi,id_lokasi,98,0,0,0,0,True
2,dim_pelanggan,id_pelanggan,1000,0,0,0,71,True
3,dim_produk,id_produk,50,0,0,0,0,True
4,dim_status_transaksi,id_status,5,0,0,0,0,True
5,dim_waktu,id_waktu,1096,0,0,0,26,True
6,fact_transaksi_layanan,id_fakta,25926,0,0,0,0,True


## Validasi Foreign Key Fakta ke Dimensi

In [8]:
fk_config = [
    ('id_waktu', dim_waktu_clean, 'id_waktu', 'dim_waktu'),
    ('id_pelanggan', dim_pelanggan_clean, 'id_pelanggan', 'dim_pelanggan'),
    ('id_produk', dim_produk_clean, 'id_produk', 'dim_produk'),
    ('id_lokasi', dim_lokasi_clean, 'id_lokasi', 'dim_lokasi'),
    ('id_channel', dim_channel_clean, 'id_channel', 'dim_channel_pembelian'),
    ('id_status', dim_status_clean, 'id_status', 'dim_status_transaksi'),
]

fk_validation = []
invalid_fk_mask = pd.Series(False, index=fact_clean.index)

for fact_column, dim_df, dim_column, dim_table in fk_config:
    valid_keys = set(dim_df[dim_column].dropna())
    current_invalid_mask = ~fact_clean[fact_column].isin(valid_keys)
    invalid_fk_mask = invalid_fk_mask | current_invalid_mask
    invalid_values = sorted(fact_clean.loc[current_invalid_mask, fact_column].dropna().astype(str).unique().tolist())

    fk_validation.append({
        'fact_table': 'fact_transaksi_layanan',
        'fact_column': fact_column,
        'dimension_table': dim_table,
        'dimension_column': dim_column,
        'invalid_row_count': int(current_invalid_mask.sum()),
        'invalid_values_sample': ', '.join(invalid_values[:10]),
    })

fk_validation_df = pd.DataFrame(fk_validation)
fk_validation_df['is_fk_valid'] = fk_validation_df['invalid_row_count'] == 0

rejected_fact_fk = fact_clean[invalid_fk_mask].copy()
fact_final = fact_clean[~invalid_fk_mask].copy()

display(fk_validation_df)
print(f'Baris fakta valid: {len(fact_final):,}')
print(f'Baris fakta ditolak karena FK tidak valid: {len(rejected_fact_fk):,}')
display(rejected_fact_fk.head())

,fact_table,fact_column,dimension_table,dimension_column,invalid_row_count,invalid_values_sample,is_fk_valid
0,fact_transaksi_layanan,id_waktu,dim_waktu,id_waktu,0,,True
1,fact_transaksi_layanan,id_pelanggan,dim_pelanggan,id_pelanggan,0,,True
2,fact_transaksi_layanan,id_produk,dim_produk,id_produk,0,,True
3,fact_transaksi_layanan,id_lokasi,dim_lokasi,id_lokasi,544,"100, 99",False
4,fact_transaksi_layanan,id_channel,dim_channel_pembelian,id_channel,0,,True
5,fact_transaksi_layanan,id_status,dim_status_transaksi,id_status,0,,True


Baris fakta valid: 25,382
Baris fakta ditolak karena FK tidak valid: 544


,id_fakta,id_waktu,id_pelanggan,id_produk,id_lokasi,id_channel,id_status,nomor_transaksi,jumlah_pembelian,total_harga,jumlah_data_mb,durasi_telpon_menit,jumlah_sms
158,478,467,190,33,99,1,1,TRX004670000478,3,405000.0,44303.72,278,349
245,786,85,570,41,99,3,4,TRX000850000786,1,15000.0,23730.88,957,382
263,835,412,625,43,99,1,1,TRX004120000835,4,480000.0,24259.49,894,427
301,960,226,189,50,100,2,3,TRX002260000960,1,200000.0,34573.46,716,449
392,1192,692,642,34,99,6,1,TRX006920001192,2,400000.0,25376.14,190,452


## Validasi Nilai Numerik Fakta

In [9]:
numeric_checks = []
for column in ['jumlah_pembelian', 'total_harga', 'jumlah_data_mb', 'durasi_telpon_menit', 'jumlah_sms']:
    numeric_checks.append({
        'column_name': column,
        'null_count': int(fact_final[column].isna().sum()),
        'negative_count': int((fact_final[column] < 0).sum()),
        'min_value': fact_final[column].min(),
        'max_value': fact_final[column].max(),
    })

numeric_validation_df = pd.DataFrame(numeric_checks)
display(numeric_validation_df)

,column_name,null_count,negative_count,min_value,max_value
0,jumlah_pembelian,0,0,0.0,995.00
1,total_harga,0,0,0.0,4564225.95
2,jumlah_data_mb,0,0,0.0,13632205.08
3,durasi_telpon_menit,0,0,0.0,247872.00
4,jumlah_sms,0,0,0.0,107358.00


## Simpan Output Processed dan Validation

In [10]:
processed_tables = {
    'dim_channel_pembelian': dim_channel_clean,
    'dim_lokasi': dim_lokasi_clean,
    'dim_pelanggan': dim_pelanggan_clean,
    'dim_produk': dim_produk_clean,
    'dim_status_transaksi': dim_status_clean,
    'dim_waktu': dim_waktu_clean,
    'fact_transaksi_layanan': fact_final,
}

for table_name, df in processed_tables.items():
    output_path = PROCESSED_DIR / f'{table_name}.csv'
    df.to_csv(output_path, index=False)
    print(f'Saved: {output_path} ({len(df):,} baris)')

table_validation_df.to_csv(VALIDATION_DIR / 'transform_table_validation.csv', index=False)
fk_validation_df.to_csv(VALIDATION_DIR / 'transform_fk_validation.csv', index=False)
numeric_validation_df.to_csv(VALIDATION_DIR / 'transform_numeric_validation.csv', index=False)
rejected_fact_fk.to_csv(VALIDATION_DIR / 'rejected_fact_transaksi_layanan_fk.csv', index=False)

print('Validation reports saved.')

Saved: c:\Codingan Kuliah\SEMESTER 6\Data Warehouse\atc-data-warehouse\data\processed\dim_channel_pembelian.csv (6 baris)
Saved: c:\Codingan Kuliah\SEMESTER 6\Data Warehouse\atc-data-warehouse\data\processed\dim_lokasi.csv (98 baris)
Saved: c:\Codingan Kuliah\SEMESTER 6\Data Warehouse\atc-data-warehouse\data\processed\dim_pelanggan.csv (1,000 baris)
Saved: c:\Codingan Kuliah\SEMESTER 6\Data Warehouse\atc-data-warehouse\data\processed\dim_produk.csv (50 baris)
Saved: c:\Codingan Kuliah\SEMESTER 6\Data Warehouse\atc-data-warehouse\data\processed\dim_status_transaksi.csv (5 baris)
Saved: c:\Codingan Kuliah\SEMESTER 6\Data Warehouse\atc-data-warehouse\data\processed\dim_waktu.csv (1,096 baris)
Saved: c:\Codingan Kuliah\SEMESTER 6\Data Warehouse\atc-data-warehouse\data\processed\fact_transaksi_layanan.csv (25,382 baris)
Validation reports saved.


## Ringkasan Akhir Transformasi

In [11]:
summary_df = pd.DataFrame({
    'table_name': list(processed_tables.keys()),
    'processed_row_count': [len(df) for df in processed_tables.values()],
    'processed_column_count': [len(df.columns) for df in processed_tables.values()],
})

display(summary_df)

final_fk_valid = True
for fact_column, dim_df, dim_column, dim_table in fk_config:
    final_fk_valid = final_fk_valid and fact_final[fact_column].isin(set(dim_df[dim_column].dropna())).all()

print('Status PK valid semua:', bool(table_validation_df['is_pk_valid'].all()))
print('Status FK fact_final valid semua:', bool(final_fk_valid))
print('Jumlah baris fakta final:', f'{len(fact_final):,}')
print('Jumlah baris fakta rejected:', f'{len(rejected_fact_fk):,}')

,table_name,processed_row_count,processed_column_count
0,dim_channel_pembelian,6,3
1,dim_lokasi,98,7
2,dim_pelanggan,1000,7
3,dim_produk,50,6
4,dim_status_transaksi,5,3
5,dim_waktu,1096,6
6,fact_transaksi_layanan,25382,13


Status PK valid semua: True
Status FK fact_final valid semua: True
Jumlah baris fakta final: 25,382
Jumlah baris fakta rejected: 544
